# T7 - B-mode Formation (standalone)
Input: reconstructed RF `(depth x lines)` from T6. Steps: Hilbert envelope -> 20 log10 -> clip to a dynamic range -> normalise to [0, 1].

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)

In [ ]:
import receiver_beamforming as rb
from bmode_formation import bmode_formation, DEFAULT_DYNAMIC_RANGE_DB
from scipy.signal import hilbert
print('default dynamic range:', DEFAULT_DYNAMIC_RANGE_DB, 'dB')

## Input: beamformed RF
Synthetic three-target data is beamformed with module 03 so this notebook runs alone. Replace `rf` with your own `(depth x lines)` array if you have one.

In [ ]:
c, fs, f0, n = 1540.0, 40e6, 5e6, 128
ex = (np.arange(n) - (n - 1) / 2) * c / f0 / 2
t = np.arange(-1.2e-6, 50e-6, 1 / fs)
rx = np.zeros((n, t.size))
for x, z in [(-3e-3, 12e-3), (0.0, 22e-3), (3e-3, 32e-3)]:
    tau = z / c + np.hypot(x - ex, z) / c
    d = t[None, :] - tau[:, None]
    rx += np.sin(2*np.pi*f0*d) * np.exp(-0.5 * (d / (3/(2.5*f0)))**2)
print("synthetic channel data:", rx.shape)
bf = rb.beamform(rb.make_receiver_input(rx, t, ex, fs_hz=fs, c0=c, f0_hz=f0), rb.RxConfig())
rf, z_axis, x_lines = bf.beamformed_rf, bf.z_axis, bf.x_lines
print('rf', rf.shape)

## Envelope and log compression

In [ ]:
env = np.abs(hilbert(rf, axis=0))
fig, ax = plt.subplots(1, 3, figsize=(14, 5)); ext = [x_lines[0]*1e3, x_lines[-1]*1e3, z_axis[-1]*1e3, z_axis[0]*1e3]
ax[0].imshow(rf, cmap='seismic', aspect='auto', extent=ext, vmin=-np.abs(rf).max(), vmax=np.abs(rf).max()); ax[0].set_title('RF')
ax[1].imshow(env, cmap='gray', aspect='auto', extent=ext); ax[1].set_title('envelope (linear)')
ax[2].imshow(bmode_formation(rf), cmap='gray', aspect='auto', extent=ext, vmin=0, vmax=1); ax[2].set_title('B-mode [0,1]')
plt.tight_layout(); plt.show()

## Effect of the dynamic range

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 5))
for a, dr in zip(ax, (30, 50, 70)):
    a.imshow(bmode_formation(rf, dr), cmap='gray', aspect='auto', extent=ext, vmin=0, vmax=1); a.set_title(f'{dr} dB')
plt.tight_layout(); plt.show()

In [ ]:
bmode = bmode_formation(rf)
print('shape', bmode.shape, '| range', bmode.min(), bmode.max())